In [ ]:
!pip install \
  torch \
  transformers==4.36.2 \
  accelerate==0.26.1 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece


  Using cached safetensors-0.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.8 kB)
  Using cached fsspec-2023.10.0-py3-none-any.whl.metadata (6.8 kB)
Using cached safetensors-0.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.3 MB)
Using cached fsspec-2023.10.0-py3-none-any.whl (166 kB)
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.2 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3

In [ ]:
import os
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

from google.colab import drive
drive.mount("/content/drive")

output_dir = "/content/drive/MyDrive/CNN_MirageNews"
os.makedirs(output_dir, exist_ok=True)
print("Output dir:", output_dir)


Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output dir: /content/drive/MyDrive/CNN_MirageNews


In [ ]:
dataset = load_dataset("anson-huang/mirage-news")
dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 2500
    })
    test1_nyt_mj: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test2_bbc_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test3_cnn_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test4_bbc_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test5_cnn_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
})

In [ ]:
model_name = "microsoft/resnet-18"

image_processor = AutoImageProcessor.from_pretrained(model_name)

model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.
Some weights of ResNetForImageClassification were not initialized from the model checkpoint at microsoft/resnet-18 and are newly initialized because the shapes did not match:
- classifier.1.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.1.weight: found shape torch.Size([1000, 512]) in the checkpoint and torch.Size([2, 512]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from PIL import Image
import torch


def preprocess(example):
    image = example["image"]

    if image is None or not isinstance(image, Image.Image):
        image = Image.new("RGB", (224, 224), (0, 0, 0))
    else:
        if image.mode != "RGB":
            image = image.convert("RGB")

    inputs = image_processor(
        image,
        return_tensors="pt"
    )

    return {
        "pixel_values": inputs["pixel_values"][0],
        "labels": example["label"]
    }


In [ ]:
encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,

    num_train_epochs=5,
    per_device_train_batch_size=16,      # 👈 giống CLIP
    per_device_eval_batch_size=32,

    learning_rate=2e-5,                  # 👈 giống CLIP/BERT
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:449: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.370800,0.122360,0.954400,0.941680,0.968800,0.955047,0.993547
2,0.136100,0.084417,0.965600,0.959716,0.972000,0.965819,0.995797
3,0.092000,0.071636,0.974800,0.972908,0.976800,0.974850,0.996755
4,0.049600,0.073984,0.972000,0.966772,0.977600,0.972156,0.996535
5,0.034500,0.074464,0.972800,0.963893,0.982400,0.973059,0.996806


TrainOutput(global_step=3125, training_loss=0.12104124130249024, metrics={'train_runtime': 6088.5458, 'train_samples_per_second': 8.212, 'train_steps_per_second': 0.513, 'total_flos': 5.047597320192e+17, 'train_loss': 0.12104124130249024, 'epoch': 5.0})

In [ ]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.0716356411576271, 'eval_accuracy': 0.9748, 'eval_precision': 0.9729083665338646, 'eval_recall': 0.9768, 'eval_f1': 0.9748502994011976, 'eval_auc': 0.9967552000000001, 'eval_runtime': 236.719, 'eval_samples_per_second': 10.561, 'eval_steps_per_second': 0.334, 'epoch': 5.0}


In [ ]:
trainer.save_model(output_dir)
image_processor.save_pretrained(output_dir)

print("Model saved to:", output_dir)


Model saved to: /content/drive/MyDrive/CNN_MirageNews
